# MCDM Weight Sensitivity — CO₂ Storage Sites

## Purpose and research question

**How much do decisions about the preferred CO₂ storage site depend on the weights assigned to the evaluation criteria?**

Multi-criteria decision-making (MCDM) combines the performance of different alternatives on several criteria. A weighted-sum model yields a ranking, but the result may change when decision-makers give different priorities to storage capacity, injectivity and storage cost. **Sensitivity analysis** investigates this dependence instead of treating a single ranking as definitive.

This notebook uses **One-at-a-Time (OAT) weight sensitivity analysis** to (1) determine an initial ranking, (2) vary the weight of each criterion separately, (3) recompute all site scores and rankings, and (4) identify ranges of weights where the preferred alternative stays the same or changes.

**What does robustness mean?** A decision is *robust to the tested changes* if the same alternative remains preferred across those weight scenarios. A decision is *sensitive* when changing the weights causes another alternative to become preferred. Robustness is always relative to the chosen criteria, score rules, tested weights and decision model: **it does not demonstrate geological certainty or site safety**.

### Method in brief

The baseline is a weighted additive model: $S_i = \sum_j w_j s_{ij}$. The selected weight is changed from 0% to 100%, while the other weights are proportionally redistributed so that the total always remains 100%. This produces **three distinct OAT experiments**, one for each criterion.

A *first-place tie* means two or more sites have equal highest overall scores. A tie does not mean they are equally safe, equally certain or physically equivalent. The winner map displays such events in turquoise only when a tie occurs at one of the sampled weights.

### Source and data

Methodological foundation: **Marco Dean (2022), *A Practical Guide to Multi-Criteria Analysis*, Section 5.8, pp. 69–71**, particularly the hypothetical one-weight sensitivity example in Figure 31 (p. 70). **All site scores, weights and plotted results in this notebook are fictional teaching data, not Dean's numerical example or measured CO₂-site evidence.**

**How to run:** Select **Runtime → Run all** in Google Colab. Code cells are collapsed by default; click **Show code** to view them. Edit the input scores or weights and rerun to update calculations and figures.

In [ ]:
#@title Import libraries { display-mode: "form" }
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


## 1. What alternatives, criteria and assumptions are being evaluated?

### Objective

Define the alternatives and their initial performance scores, along with the criterion weights used to establish a reference case. Without a transparent starting point, changes in subsequent results cannot be interpreted.

**Table 1** contains three *fictional* storage sites and three illustrative criteria. Scores use a common desirability scale: **1 = least desirable and 5 = most desirable**. In particular, the *Storage Cost* column is a converted desirability score: a larger score represents a **lower actual cost**, not a greater cost in €/t.

**Table 2** contains the initial criterion weights: 40% storage capacity, 35% injectivity and 25% storage cost. They sum to 100%.

### How are weights changed?

In each OAT experiment only one criterion weight is independently selected. The remaining two weights preserve their original ratio. For example, when *Storage Capacity* is set to 0%, the other weights become **58.33% Injectivity and 41.67% Storage Cost** because 35/(35 + 25) and 25/(35 + 25) are their original proportions. When capacity returns to 40%, the original 35%/25% allocation is restored. The same procedure is repeated independently for Injectivity and Storage Cost.

### Interpretation

These inputs are *assumptions*, not site observations. A different scoring system or weighting scheme can produce a different ranking. When applying the method to actual CO₂ storage sites, the origin and justification of each score and weight must be documented.

In [ ]:
#@title Editable inputs { display-mode: "form" }
capacity_weight_pct = 40 #@param {type:"number"}
injectivity_weight_pct = 35 #@param {type:"number"}
cost_weight_pct = 25 #@param {type:"number"}
sensitivity_step_pct = 5 #@param {type:"number"}

# Fictional scores: each row is a site; each column is a criterion.
# You can replace these scores with validated 1–5 desirability scores.
site_scores = pd.DataFrame(
    {
        "Storage capacity": [5, 3, 4],
        "Injectivity": [3, 5, 2],
        "Storage cost": [2, 4, 5],
    },
    index=["Site A", "Site B", "Site C"],
)
criteria = list(site_scores.columns)
weights_pct = np.array([capacity_weight_pct, injectivity_weight_pct, cost_weight_pct], dtype=float)
if np.any(~np.isfinite(weights_pct)) or np.any(weights_pct < 0) or not np.isclose(weights_pct.sum(), 100):
    raise ValueError("Baseline weights must be finite, nonnegative, and sum to 100%.")
numeric_scores = site_scores.to_numpy(dtype=float)
if not np.all(np.isfinite(numeric_scores)) or not np.all((numeric_scores >= 1) & (numeric_scores <= 5)):
    raise ValueError("All scores must be numeric and between 1 and 5.")
if sensitivity_step_pct <= 0 or sensitivity_step_pct > 100 or not np.isclose(100 / sensitivity_step_pct, round(100 / sensitivity_step_pct)):
    raise ValueError("Sensitivity step must divide 100 exactly (e.g. 1, 2, 5, 10).")
weights = weights_pct / 100
display(Markdown("**Table 1. Fictional site-performance scores on a 1–5 desirability scale.**"))
display(site_scores)
display(Markdown("*Note: 5 indicates highest desirability; for Storage Cost, 5 represents lower cost.*"))
display(Markdown("**Table 2. Baseline weights assigned to each evaluation criterion.**"))
display(pd.DataFrame({"Criterion": criteria, "Weight (%)": weights_pct}))
display(Markdown("*Note: criterion weights must total 100%.*"))
print("Editable inputs validated successfully.")


## 2. Which site is preferred under the initial weights?

### Objective

Establish a **baseline ranking** against which all sensitivity scenarios can be compared.

The weighted sum is $S_i = \sum_j w_j s_{ij}$, where $s_{ij}$ is the desirability score of site *i* on criterion *j* and the weights $w_j$ sum to one. **Table 3** contains the resulting totals and ranks. **Figure 1** visualizes the same scores.

### Interpretation of the baseline result

Under the default fictional inputs, **Site B scores 3.95** and ranks first. **Sites A and C both score 3.55**, so they are tied for second place. They must not be interpreted as distinctly ranked because the display happens to list A before C.

The baseline identifies the preferred option **under the specified weights only**. It does not tell us whether the preference survives changes in those weights; subsequent sections address that question.

In [ ]:
#@title Baseline ranking { display-mode: "form" }
def weighted_results(score_table, weight_vector):
    if not np.isclose(np.sum(weight_vector), 1) or np.any(weight_vector < -1e-12):
        raise ValueError("Weights must sum to one and be nonnegative.")
    values = score_table.to_numpy(dtype=float) @ np.asarray(weight_vector, dtype=float)
    result = pd.DataFrame({"Site": score_table.index, "Total score": values})
    result["Rank"] = result["Total score"].rank(method="min", ascending=False).astype(int)
    return result.sort_values(["Total score", "Site"], ascending=[False, True]).reset_index(drop=True)

def preference_signature(table):
    """Pairwise order with numerical ties handled explicitly."""
    scores = table.set_index("Site")["Total score"]
    names = sorted(scores.index)
    return tuple(0 if np.isclose(scores[x], scores[y], atol=1e-9, rtol=0)
                 else (1 if scores[x] > scores[y] else -1)
                 for i, x in enumerate(names) for y in names[i+1:])

baseline = weighted_results(site_scores, weights)
display(Markdown("**Table 3. Baseline ranking of the three fictional CO₂ storage sites.**"))
display(baseline.round(3))
display(Markdown("*Note: total scores are dimensionless weighted desirability indices; equal scores share the same rank.*"))
ax = baseline.plot.bar(x="Site", y="Total score", legend=False, figsize=(7, 4))
ax.set_ylabel("Weighted score (1–5)")
ax.set_ylim(0, 5)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
display(Markdown("**Figure 1.** Baseline weighted desirability scores for the three fictional storage sites under the selected criterion weights. The highest score indicates the preferred site under the assumed model; ties represent equal total desirability."))


## 3. How are alternative weight scenarios calculated?

### Objective

Create a reproducible set of weight scenarios and calculate the score of every site under each one, rather than relying on a single weighting scheme.

With the default **5-percentage-point step**, each of the three criteria is tested at 0%, 5%, 10%, …, 100%: **21 scenarios per criterion, or 63 scenarios in total**.

### How to read the sensitivity results

**Table 4** contains all these scenarios:
- **Criterion varied:** which single criterion is being tested in that particular OAT experiment.
- **Tested weight (%):** the selected value for that criterion.
- **Three weight columns:** the new full allocation, always totaling 100%.
- **Site A/B/C score:** each site's *overall* recalculated weighted score.
- **Winner(s):** all sites tied for highest overall score, if any.
- **Display order:** scores ordered from highest to lowest, with ties only alphabetically ordered for presentation.
- **Order changed vs baseline:** whether any pairwise preference or tie status differs from the baseline.

For example, the row testing **0% Storage Capacity** has **58.33% Injectivity** and **41.67% Storage Cost**. These are not independent increases or measurements: they result from proportional redistribution of the original remaining weights.

### Interpretation

Each row asks: *If the decision-maker selected this weight, which site would be preferred?* The rows for Injectivity and Storage Cost are separate experiments and are not variations performed simultaneously with Capacity. The sensitivity curves in the next section visualize the recalculated site-score columns across the complete table.

In [ ]:
#@title Calculate all OAT scenarios { display-mode: "form" }
def rescale_weights(base_weights, index, tested_weight):
    if not 0 <= tested_weight <= 1:
        raise ValueError("Tested weight must be between 0 and 1.")
    original_rest = 1 - base_weights[index]
    if original_rest <= 1e-12:
        if np.isclose(tested_weight, 1):
            result = np.zeros_like(base_weights, dtype=float)
            result[index] = 1
            return result
        raise ValueError("Cannot proportionally rescale other zero baseline weights.")
    result = np.asarray(base_weights, dtype=float) * ((1 - tested_weight) / original_rest)
    result[index] = tested_weight
    return result

rows = []
steps = int(round(100 / sensitivity_step_pct))
baseline_signature = preference_signature(baseline)
for j, criterion in enumerate(criteria):
    for k in range(steps + 1):
        tested_weight = k / steps
        trial_weights = rescale_weights(weights, j, tested_weight)
        trial = weighted_results(site_scores, trial_weights)
        best = trial["Total score"].max()
        winners = sorted(trial.loc[np.isclose(trial["Total score"], best), "Site"].tolist())
        rows.append({
            "Criterion varied": criterion,
            "Tested weight (%)": 100 * tested_weight,
            **{f"{name} weight (%)": 100 * trial_weights[i] for i, name in enumerate(criteria)},
            **{f"{site} score": float(trial.loc[trial["Site"].eq(site), "Total score"].iloc[0]) for site in site_scores.index},
            "Winner(s)": " / ".join(winners),
            "Display order": " > ".join(trial["Site"]),
            "Order changed vs baseline": preference_signature(trial) != baseline_signature,
        })
sensitivity = pd.DataFrame(rows)
display(Markdown("**Table 4. One-at-a-time sensitivity results across all three criteria and tested weights.**"))
try:
    from google.colab import data_table
    display(data_table.DataTable(sensitivity.round(2), include_index=False, num_rows_per_page=15))
except ImportError:
    display(sensitivity.round(2))
display(Markdown("*Note: each row changes one criterion weight, adjusts remaining weights proportionally, and recalculates scores and the preferred alternative.*"))
print(f"Evaluated {len(sensitivity)} scenarios ({len(criteria)} criteria × {steps+1} tested weights).")


## 4. At which weights do the site preferences change?

### Objective

Visualize how each site's **total weighted score** responds to changes in the weight assigned to an individual criterion. This helps identify crossing points where the ranking may change.

**Figure 2** contains three OAT panels, one for **Storage Capacity**, one for **Injectivity** and one for **Storage Cost**. In every panel, the x-axis is the weight of that panel's criterion; the y-axis is the *overall weighted score* across all three criteria. The other two weights are proportionally adjusted. Blue, orange and green show Sites A, B and C; the dashed gray line marks the baseline weight.

A crossing between two curves means the two sites have equal total scores **at that weight**. If those sites are jointly highest, the intersection marks a transition in the preferred option. Crossings between lower-ranked sites may change the ordering without changing the winner.

The approach is inspired by Dean (2022), **Section 5.8, pp. 69–70, Figure 31 (p. 70)**. The plots and numerical results here use the fictional example.

In [ ]:
#@title Figure 2 — Sensitivity curves { display-mode: "form" }
fig, axes = plt.subplots(1, len(criteria), figsize=(17, 4.5), sharey=True)
for j, (criterion, ax) in enumerate(zip(criteria, axes)):
    subset = sensitivity.loc[sensitivity["Criterion varied"] == criterion]
    for site in site_scores.index:
        ax.plot(subset["Tested weight (%)"], subset[f"{site} score"], label=site, linewidth=2)
    ax.axvline(weights_pct[j], linestyle="--", color="gray", linewidth=1, label="Baseline")
    ax.set_xlabel(f"{criterion} weight (%)")
    ax.set_xlim(0, 100)
    ax.grid(alpha=.25)
    ax.legend(loc="best", fontsize=9, framealpha=0.9)
axes[0].set_ylabel("Weighted overall score (1–5)")

plt.tight_layout()
plt.show()
display(Markdown("**Figure 2.** One-at-a-time (OAT) weight sensitivity of total weighted desirability scores for Sites A, B and C. In each panel, the indicated criterion weight varies from 0% to 100% while the other two weights are adjusted proportionally. Dashed vertical lines indicate baseline weights; curve intersections identify equal total scores and potential ranking changes. Methodological inspiration: Dean (2022), Section 5.8, Figure 31 (p. 70); the results shown here use our fictional data."))


### Interpretation of Figure 2

**Storage Capacity.** With the default inputs, **Site B** is preferred below a 50% capacity weight; **A and B tie at 50%**; and **Site A** is preferred above 50%. This reflects Site A's highest capacity desirability score (5/5). The baseline capacity weight (40%) lies in the B-preferred range.

**Injectivity.** **Site C** leads when injectivity is below 25%; **B and C tie at 25%**; and **Site B** leads above that threshold. The baseline injectivity weight (35%) favors B, which has the highest injectivity score (5/5).

**Storage Cost.** At very low cost weight, **Site A** leads; **Site B** leads at intermediate weights; and **Site C** becomes preferred when cost receives sufficiently high emphasis. For the default data, the winner transitions occur near **6.25%** and **46.43%** cost weight. Site C benefits from the highest cost desirability score (5/5), representing a relatively low modeled cost.

**Scientific implication.** Site B is the preferred alternative at the baseline weights, but not under all weight assumptions. Thus a ranking alone hides trade-offs. These crossover values are *specific to the fictional default inputs*; changing the scores or weights changes the conclusions.

## 5. How robust is the ranking to changes in weights?

### Objective

Assess **ranking robustness**, meaning how consistently the baseline ordering and the preferred first-place alternative are preserved as the tested weights change.

A **robust first-place decision** is one in which the same site remains best across the selected scenarios. A **sensitive decision** changes its winner after changes in the weighting assumptions. This is a measure of **preference stability**, not geological confidence.

**Table 5** and **Figure 3** summarize two different measures:

- **Displayed ranking changes:** at least one pair of sites changes relative order or changes between a tie and a strict preference, even when the winner remains unchanged.
- **First-place set changes:** the set of best-ranked sites differs from the baseline. This includes a new sole winner or a new tie for first place.

The bars show the **share of tested weight settings** causing each change, not the probability of those changes occurring in practice. With the default grid, each criterion contributes 21 tested settings. Importantly, Sites A and C begin tied for second; breaking this tie affects the first measure but not necessarily the second.

In [ ]:
#@title Robustness summary and bar chart { display-mode: "form" }
baseline_best = baseline["Total score"].max()
baseline_winners = " / ".join(sorted(
    baseline.loc[np.isclose(baseline["Total score"], baseline_best), "Site"].tolist()
))
sensitivity["First-place set changed"] = sensitivity["Winner(s)"] != baseline_winners
summary = (
    sensitivity.groupby("Criterion varied", sort=False)
    .agg(
        Tested_scenarios=("Order changed vs baseline", "size"),
        Ranking_changes=("Order changed vs baseline", "sum"),
        First_place_changes=("First-place set changed", "sum"),
    )
    .reset_index()
)
summary["Ranking changes (%)"] = 100 * summary["Ranking_changes"] / summary["Tested_scenarios"]
summary["First-place changes (%)"] = 100 * summary["First_place_changes"] / summary["Tested_scenarios"]
print("Baseline winner(s):", baseline_winners)
display(Markdown("**Table 5. Summary of ranking and first-place sensitivity under one-at-a-time weight variation.**"))
display(summary.round(1))
display(Markdown("*Note: percentages indicate the proportion of tested weight settings that change pairwise ranking or first-place site(s), not probabilities of geological outcomes.*"))

ax = summary.plot.bar(
    x="Criterion varied",
    y=["Ranking changes (%)", "First-place changes (%)"],
    figsize=(9, 4.5),
    rot=0,
)
ax.set_ylabel("Share of tested weights (%)")
ax.set_xlabel("")
ax.set_ylim(0, 105)
ax.set_title("Sensitivity to each criterion's weight")
ax.legend(["Displayed ranking changes", "First-place set changes"])
plt.tight_layout()
plt.show()
display(Markdown("**Figure 3.** Percentage of tested weight settings that alter the pairwise ranking or change the set of first-place sites, evaluated separately for each criterion. These percentages describe the specified OAT grid rather than likelihoods of real-world outcomes."))



### Interpretation of Table 5 and Figure 3

The two metrics answer distinct questions: *Does the complete ranking remain unchanged?* and *Does the preferred alternative remain unchanged?* The first is stricter because changes among lower-ranked sites also count; the second directly concerns the site-selection decision.

For the default example, changes in **Capacity** and **Cost** across the full 0–100% range produce more preference reversals than changes in **Injectivity**. This indicates that the result is particularly conditional on how those criteria are weighted **over the grid tested**. It does **not** imply these criteria are more geologically uncertain or that each tested weight is equally plausible.

To make a defensible decision in an applied study, analysts should also examine a narrower range of weights supported by stakeholder judgments or an explicit weighting-elicitation process.

## 6. Under which weighting assumptions does each site become the preferred option?

### Objective

Present the **first-place stability map** (Figure 4), which identifies the preferred alternative over the sampled range of criterion weights. This helps reveal whether the baseline recommendation is stable or conditional on a particular preference structure.

### How to read the map

Each row is **one independent OAT experiment**. The horizontal axis gives the weight of the criterion named on that row, from 0% to 100%. At every sampled weight, the map shows **the site with the highest overall weighted score**. The map is not geographical: the colors represent ranking outcomes, not subsurface locations or levels of geological risk.

### What does the turquoise “Tie” area mean?

**Turquoise indicates that two or more sites share exactly the highest overall score at a sampled weight.** No unique site is preferred by this weighted-sum model at that point.

For the default data:
- At **50% Storage Capacity**, Sites **A and B tie for first place**, marking the boundary between B-preferred and A-preferred weight settings.
- At **25% Injectivity**, Sites **B and C tie for first place**, marking the boundary between C-preferred and B-preferred settings.

**The colored rectangle is not a continuous tie zone.** It has visible width because the heatmap represents discrete weights in 5% steps. Equality occurs at the **sampled weight**, not necessarily throughout the entire painted rectangle. A tie at an unsampled weight can be missed; the continuous score curves in Figure 2 help identify those intersections. Ties between two *lower-ranked* sites do not appear as turquoise because the map only shows first-place ties.

In [ ]:
#@title First-place stability map { display-mode: "form" }
winner_labels = list(site_scores.index) + ["Tie"]
winner_to_code = {name: i for i, name in enumerate(winner_labels)}
winner_matrix = []
for criterion in criteria:
    subset = sensitivity.loc[sensitivity["Criterion varied"] == criterion]
    winner_matrix.append([
        winner_to_code[w] if w in winner_to_code else winner_to_code["Tie"]
        for w in subset["Winner(s)"]
    ])
winner_matrix = np.asarray(winner_matrix)
fig, ax = plt.subplots(figsize=(12, 3.5))
from matplotlib.colors import BoundaryNorm
cmap = plt.get_cmap("tab10", len(winner_labels))
norm = BoundaryNorm(np.arange(-0.5, len(winner_labels) + 0.5, 1), cmap.N)
im = ax.imshow(winner_matrix, aspect="auto", interpolation="nearest", cmap=cmap, norm=norm)
ax.set_yticks(range(len(criteria)), criteria)
ax.set_xticks(range(0, steps + 1, max(1, steps // 10)))
ax.set_xticklabels([f"{100 * k / steps:.0f}%" for k in range(0, steps + 1, max(1, steps // 10))])
ax.set_xlabel("Tested criterion weight")

cbar = fig.colorbar(im, ax=ax, ticks=range(len(winner_labels)), pad=0.02)
cbar.ax.set_yticklabels(winner_labels)
plt.tight_layout()
plt.show()
display(Markdown("**Figure 4.** First-place stability map for three independent OAT experiments. Each row varies the named criterion’s weight; each color shows the top-ranked site at a sampled weight. Turquoise **Tie** cells denote co-winners with equal highest overall scores at that specific tested weight (default: A–B at 50% capacity; B–C at 25% injectivity). The colored cell width reflects the 5% sampling interval, **not a continuous tie zone**. A color transition indicates a switch in the preferred site."))


### Interpretation of Figure 4

For the fictional baseline, **Site B** is preferred at 40% Capacity, 35% Injectivity and 25% Cost. The map demonstrates that this result is **conditional rather than universally stable**.

Increasing the importance of Capacity beyond 50% favors **Site A**. Setting Injectivity below 25% favors **Site C**, whereas increasing Injectivity above that threshold favors **Site B**. For Storage Cost, the preferred option transitions from **A** at low weights, to **B** at intermediate weights, and to **C** at high weights.

The first-place map therefore provides a useful complement to the score curves: it makes **winner reversals** and **co-winner thresholds** immediately visible.

### Discussion for reporting

*Under the baseline weighting assumptions, Site B achieved the highest total desirability score. However, the one-at-a-time sensitivity analysis demonstrated that the preferred option changed when the relative importance of individual criteria was altered. The first-place stability map identified transitions between sites, including first-place ties at specific weight thresholds. The results emphasize that site preference should be reported together with the tested weighting assumptions and ranking robustness, rather than as an unconditional selection.*

This interpretation applies **only to the illustrative data**. An applied CO₂ storage-site assessment would require validated technical indicators, defensible scoring functions and weights, and non-compensatory screening of safety and regulatory requirements.

## 7. How much does the full ranking change? — Spearman correlation

### Objective

Complement the existing OAT analysis by comparing the **baseline ranking** with the ranking in **every OAT scenario**. A change in the first-place site does not tell us how much the *entire* ordering changed.

Spearman's rank correlation (ρ) measures the similarity of two sets of ranks: **+1** means identical ordering, **−1** means completely reversed ordering, and intermediate values indicate partial agreement. **Figure 5** shows ρ against each varied weight; **Table 6** shows the corresponding rankings and correlations.

**Important:** rankings are calculated from weighted scores, not alphabetical display order. **Ties receive average ranks**. If all sites tie in a scenario, ρ is undefined (NaN), not zero. With only three fictional sites, ρ takes a small set of possible values; interpret it descriptively rather than as a significance test.

### How to interpret

- **ρ = +1:** the three sites retain the same rank ordering.
- **ρ below +1:** at least part of the ordering has changed; inspect Table 6 to see which sites moved.
- **ρ = −1:** a complete reversal, possible when neither ranking contains ties.
- Compare **ρ** with **First-place set changed**: they measure different aspects of robustness.

This is **Spearman between rankings**, not Spearman between a weight and a site's position. The latter is a separate analysis.


In [ ]:
#@title Figure 5 and Table 6 — Spearman similarity to baseline { display-mode: "form" }
site_names = list(site_scores.index)
baseline_scores = baseline.set_index("Site").reindex(site_names)["Total score"]
baseline_ranks = baseline_scores.rank(ascending=False, method="average")
baseline_best = baseline_scores.max()
baseline_winners_set = set(baseline_scores.index[np.isclose(baseline_scores, baseline_best)])

spearman_rows = []
for _, scenario in sensitivity.iterrows():
    scenario_scores = pd.Series(
        {site: float(scenario[f"{site} score"]) for site in site_names}, dtype=float
    )
    scenario_ranks = scenario_scores.rank(ascending=False, method="average")
    # Spearman = Pearson correlation of average ranks (handles ties).
    rho = baseline_ranks.corr(scenario_ranks, method="pearson")
    scenario_winners = set(scenario_scores.index[np.isclose(scenario_scores, scenario_scores.max())])
    spearman_rows.append({
        "Criterion varied": scenario["Criterion varied"],
        "Tested weight (%)": scenario["Tested weight (%)"],
        "Spearman rho vs baseline": rho,
        "First-place set changed": scenario_winners != baseline_winners_set,
        **{f"{site} baseline rank": baseline_ranks.loc[site] for site in site_names},
        **{f"{site} OAT rank": scenario_ranks.loc[site] for site in site_names},
    })

spearman_results = pd.DataFrame(spearman_rows)


### Figure 5 — Spearman versus OAT weight (shown before the long table)

The **blue line** shows Spearman's ρ between each scenario ranking and the baseline ranking. **The grey dashed vertical line** marks the original criterion weight: Capacity **40%**, Injectivity **35%**, and Cost **25%** with default inputs. The **horizontal dotted line at ρ = 1** marks identical ranks. The X-axis displays explicit 5%-interval ticks.

**How to read the three panels (default fictional inputs):**

- **Storage capacity:** Spearman is about **0.87** at most weights from 0% to 50%, reaches **1** at the 40% baseline, falls to **0** near 55%, and to about **−0.87** from 60%. Large capacity weights lead to a substantially different ordering.
- **Injectivity:** Spearman is about **−0.87** at 0–15%, reaches **0** at 20%, and is about **0.87** at most weights from 25% upwards; it is **1** at the 35% baseline. Very low injectivity weights strongly change the ordering.
- **Storage cost:** Spearman is **0** at 0–5%, about **0.87** at most weights from 10% to 45%, **1** at the 25% baseline, and **0** from 50% onwards. High cost weights alter the ordering, but ρ = 0 does **not** mean all sites tie.

**Why is the baseline sometimes the only point at ρ = 1?** The baseline has a tie in its weighted scores, so we use **average ranks for ties**. A tiny change can break the tie and yield ρ ≈ 0.87, even if the first-place site is unchanged. For only three sites, ρ is discrete and should be interpreted alongside the individual site ranks and winner changes in Table 6.

**Open the plot separately:** The next cell displays Figure 5 *before* the long table, and saves a standalone interactive HTML version. In Colab's Files panel (folder icon), locate `figure_5_spearman.html`, use its three-dot menu to download it, then open it in your browser as a separate window or tab. This avoids scrolling through Table 6. Rerun the notebook after changing inputs.


In [ ]:
#@title Figure 5 — Spearman plot (before Table 6) { display-mode: "form" }
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig5 = make_subplots(rows=1, cols=len(criteria), subplot_titles=criteria, shared_yaxes=True)
for j, criterion in enumerate(criteria, start=1):
    subset = spearman_results.loc[spearman_results["Criterion varied"] == criterion]
    fig5.add_trace(go.Scatter(
        x=subset["Tested weight (%)"], y=subset["Spearman rho vs baseline"],
        mode="lines+markers", name="Spearman rho", showlegend=(j == 1),
        hovertemplate="Weight: %{x:.0f}%<br>Spearman ρ: %{y:.3f}<extra></extra>"
    ), row=1, col=j)
    fig5.add_vline(x=float(weights_pct[j-1]), line_dash="dash", line_color="gray",
                   row=1, col=j)
    fig5.update_xaxes(title_text="Criterion weight (%)", tickmode="linear",
                      tick0=0, dtick=5, range=[0,100], tickangle=45, row=1, col=j)
    fig5.update_yaxes(range=[-1.05,1.05], dtick=0.25, row=1, col=j)
    fig5.add_hline(y=1, line_dash="dot", line_color="gray", row=1, col=j)
fig5.update_layout(title="Figure 5 — OAT ranking similarity to baseline (WSM)",
                   height=520, width=1450, hovermode="closest",
                   legend=dict(orientation="h", y=1.14))
fig5.show()
fig5.write_html("figure_5_spearman.html", include_plotlyjs=True, full_html=True)
print("Separate interactive graph saved: figure_5_spearman.html (Colab Files panel → download → open in browser)")


### Table 6 — Full OAT ranking comparison

The table below reports Spearman's ρ and each site's baseline and OAT ranks for every tested weight. **First-place set changed** separately records whether the preferred site or tied group of preferred sites changed. This table appears *after* Figure 5 so you do not need to scroll past it to find the plot.


In [ ]:
#@title Table 6 — All OAT rankings and Spearman values { display-mode: "form" }
print("Table 6 — Ranking similarity for every OAT scenario")
display(spearman_results.style.format({"Spearman rho vs baseline": "{:.3f}"}))
spearman_results.to_csv("mcdm_oat_spearman_rankings.csv", index=False)
print("Saved: mcdm_oat_spearman_rankings.csv")


## 8. What are the limitations, and which methodology supports the analysis?

### Methodological reference

Dean, M. (2022). *A Practical Guide to Multi-Criteria Analysis*. University College London. **Section 5.8, pp. 69–71; Figure 31 (p. 70); Table 18 (p. 71).**

Dean describes sensitivity analysis as a means of exploring whether alternative scores and rankings are affected by changes in model assumptions, including criterion scores and weights. The one-weight example informs the present approach; **the fictional data, proportional reweighting, figures, transition values and interpretations were developed independently for this notebook**.

### Limitations and interpretation

This analysis varies **weights**, not geological property uncertainty, measured injectivity or economic input distributions. The grid from 0% to 100% is exploratory and may extend beyond realistic stakeholder preferences. The weighted-sum model permits compensation between criteria, so unacceptable containment or regulatory risks should not be overlooked merely because a site achieves a high overall score. For an operational CO2SITE application, review scoring scales, missing data, criterion dependencies, double counting, admissibility thresholds and uncertainty before using the rankings to support investment decisions.